In [ ]:

try:
    from google.colab import files
    uploaded = files.upload()
    print("Fichiers uploadés:", list(uploaded.keys()))
except Exception as e:
    print("Hors Colab ou upload non nécessaire:", e)


Saving exemplo_arquivo_respostas.csv to exemplo_arquivo_respostas.csv
Saving conjunto_de_teste.csv to conjunto_de_teste.csv
Saving conjunto_de_treinamento.csv to conjunto_de_treinamento.csv
Fichiers uploadés: ['exemplo_arquivo_respostas.csv', 'conjunto_de_teste.csv', 'conjunto_de_treinamento.csv']


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import make_scorer

from xgboost import XGBRegressor

RANDOM_STATE = 42


In [ ]:
# --- Chemins des fichiers (adapte si besoin) ---
TRAIN_PATH = "conjunto_de_treinamento.csv"
TEST_PATH  = "conjunto_de_teste.csv"
SAMPLE_SUB_PATH = "exemplo_arquivo_respostas.csv"

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print("train shape:", train.shape)
print("test  shape:", test.shape)
print("sample_sub columns:", list(sample_sub.columns))
train.head()


train shape: (4683, 21)
test  shape: (2000, 20)
sample_sub columns: ['Id', 'preco']


,Id,tipo,bairro,tipo_vendedor,quartos,suites,vagas,area_util,area_extra,diferenciais,...,estacionamento,piscina,playground,quadra,s_festas,s_jogos,s_ginastica,sauna,vista_mar,preco
0,2000,Casa,Imbiribeira,Imobiliaria,3,3,5,223,167,piscina e copa,...,0,1,0,0,0,0,0,0,0,1000000.0
1,2001,Apartamento,Casa Amarela,Imobiliaria,4,4,2,157,0,piscina e churrasqueira,...,0,1,0,0,0,0,0,0,0,680000.0
2,2002,Apartamento,Encruzilhada,Imobiliaria,3,1,0,53,0,nenhum,...,0,0,0,0,0,0,0,0,0,450000.0
3,2003,Apartamento,Boa Viagem,Imobiliaria,4,3,2,149,0,piscina e churrasqueira,...,0,1,0,0,0,0,0,0,0,1080000.0
4,2004,Apartamento,Rosarinho,Imobiliaria,2,1,1,54,0,piscina e churrasqueira,...,0,1,0,0,0,0,0,0,0,350000.0


In [ ]:

id_col = sample_sub.columns[0]
target_col = sample_sub.columns[1]

print("id_col:", id_col)
print("target_col:", target_col)

# Cible dans le train (souvent 'preco')
if target_col not in train.columns:
    if "preco" in train.columns:
        target_col_in_train = "preco"
    else:
        raise ValueError(f"Impossible de trouver la cible. Ni '{target_col}' ni 'preco' dans train.")
else:
    target_col_in_train = target_col

print("target_col_in_train:", target_col_in_train)

X_train = train.drop(columns=[target_col_in_train])
y_train = train[target_col_in_train]

X_test = test.copy()


id_col: Id
target_col: preco
target_col_in_train: preco


In [ ]:

categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = [c for c in X_train.columns if c not in categorical_features]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

print("Num features:", len(numeric_features))
print("Cat features:", len(categorical_features))


Num features: 16
Cat features: 4


In [ ]:

y_train_log = np.log1p(y_train)


In [ ]:
# --- Scorer RMSPE ---
def rmspe_from_log(y_true_log, y_pred_log, eps=1e-8):
    y_true = np.expm1(np.asarray(y_true_log))
    y_pred = np.expm1(np.asarray(y_pred_log))
    denom = np.maximum(np.abs(y_true), eps)
    return np.sqrt(np.mean(((y_true - y_pred) / denom) ** 2))

rmspe_scorer = make_scorer(rmspe_from_log, greater_is_better=False)


In [ ]:

base_model = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

pipe = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", base_model)
])

cv = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)  # rapide


In [ ]:

param_dist = {
    "model__n_estimators": np.arange(300, 1501, 100),
    "model__learning_rate": np.logspace(np.log10(0.01), np.log10(0.2), 20),
    "model__max_depth": np.arange(3, 11),
    "model__min_child_weight": np.arange(1, 11),
    "model__subsample": np.linspace(0.6, 1.0, 9),
    "model__colsample_bytree": np.linspace(0.6, 1.0, 9),
    "model__gamma": np.linspace(0.0, 3.0, 16),
    "model__reg_alpha": np.linspace(0.0, 5.0, 11),
    "model__reg_lambda": np.linspace(0.0, 15.0, 16),
}

rs = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=40,
    scoring=rmspe_scorer,
    cv=cv,
    n_jobs=-1,
    verbose=2,
    random_state=RANDOM_STATE
)

rs.fit(X_train, y_train_log)

print("Best RMSPE (CV) after randomized:", -rs.best_score_)
print("Best params:", rs.best_params_)


Fitting 3 folds for each of 40 candidates, totalling 120 fits
Best RMSPE (CV) after randomized: 2.655310919818804
Best params: {'model__subsample': np.float64(0.8), 'model__reg_lambda': np.float64(7.0), 'model__reg_alpha': np.float64(1.5), 'model__n_estimators': np.int64(1500), 'model__min_child_weight': np.int64(7), 'model__max_depth': np.int64(6), 'model__learning_rate': np.float64(0.016048180071713387), 'model__gamma': np.float64(0.0), 'model__colsample_bytree': np.float64(1.0)}


In [ ]:

best = rs.best_params_

def clamp(x, lo, hi):
    return max(lo, min(hi, x))

def uniq_sorted(vals):
    return sorted(set(vals))

param_dist_local = {
    "model__max_depth": [d for d in uniq_sorted([best["model__max_depth"]-1,
                                                best["model__max_depth"],
                                                best["model__max_depth"]+1]) if d >= 1],
    "model__min_child_weight": uniq_sorted([clamp(best["model__min_child_weight"]-1, 1, 50),
                                           best["model__min_child_weight"],
                                           clamp(best["model__min_child_weight"]+1, 1, 50)]),
    "model__subsample": uniq_sorted([clamp(best["model__subsample"]-0.05, 0.5, 1.0),
                                    best["model__subsample"],
                                    clamp(best["model__subsample"]+0.05, 0.5, 1.0)]),
    "model__colsample_bytree": uniq_sorted([clamp(best["model__colsample_bytree"]-0.05, 0.5, 1.0),
                                           best["model__colsample_bytree"],
                                           clamp(best["model__colsample_bytree"]+0.05, 0.5, 1.0)]),
    "model__gamma": uniq_sorted([clamp(best["model__gamma"]-0.2, 0.0, 10.0),
                                 best["model__gamma"],
                                 clamp(best["model__gamma"]+0.2, 0.0, 10.0)]),
    "model__reg_alpha": uniq_sorted([clamp(best["model__reg_alpha"]-0.5, 0.0, 50.0),
                                     best["model__reg_alpha"],
                                     clamp(best["model__reg_alpha"]+0.5, 0.0, 50.0)]),
    "model__reg_lambda": uniq_sorted([clamp(best["model__reg_lambda"]-1.0, 0.0, 100.0),
                                      best["model__reg_lambda"],
                                      clamp(best["model__reg_lambda"]+1.0, 0.0, 100.0)]),
    "model__learning_rate": uniq_sorted([clamp(best["model__learning_rate"]*0.85, 0.005, 0.3),
                                        best["model__learning_rate"],
                                        clamp(best["model__learning_rate"]*1.15, 0.005, 0.3)]),
    "model__n_estimators": uniq_sorted([clamp(best["model__n_estimators"]-200, 100, 3000),
                                       best["model__n_estimators"],
                                       clamp(best["model__n_estimators"]+200, 100, 3000)]),
}

cv_fast = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

rs2 =   (
    estimator=pipe,
    param_distributions=param_dist_local,
    n_iter=18,
    scoring=rmspe_scorer,
    cv=cv_fast,
    n_jobs=-1,
    verbose=2,
    random_state=RANDOM_STATE
)

rs2.fit(X_train, y_train_log)

print("Best RMSPE (CV) after local refine:", -rs2.best_score_)
print("Best params after local refine:", rs2.best_params_)

best_model = rs2.best_estimator_


Fitting 3 folds for each of 18 candidates, totalling 54 fits
Best RMSPE (CV) after local refine: 2.6410374135210115
Best params after local refine: {'model__subsample': np.float64(0.8), 'model__reg_lambda': np.float64(6.0), 'model__reg_alpha': np.float64(1.5), 'model__n_estimators': np.int64(1700), 'model__min_child_weight': np.int64(7), 'model__max_depth': np.int64(7), 'model__learning_rate': np.float64(0.016048180071713387), 'model__gamma': 0.0, 'model__colsample_bytree': np.float64(1.0)}


In [ ]:
# --- Entraînement final + prédiction ---
best_model.fit(X_train, y_train_log)

pred_test_log = best_model.predict(X_test)
pred_test = np.expm1(pred_test_log)
pred_test = np.maximum(pred_test, 0)  # sécurité

pred_test[:5]


array([1542233.8 ,  254461.8 ,  602878.2 ,  227803.84,  371317.62],
      dtype=float32)

In [ ]:
# --- Soumission Kaggle ---
if id_col not in test.columns:
    # fallback commun
    if "row_id" in test.columns and id_col in ["Id", "id"]:
        test_for_id = test.rename(columns={"row_id": id_col})
    else:
        raise ValueError(f"Colonne ID '{id_col}' introuvable dans le test. Colonnes: {list(test.columns)}")
else:
    test_for_id = test

submission = pd.DataFrame({
    id_col: test_for_id[id_col],
    target_col: pred_test
})

submission.to_csv("submission.csv", index=False)
print("Fichier créé: submission.csv")
submission.head()


✅ Fichier créé: submission.csv


,Id,preco
0,0,1.542234e+06
1,1,2.544618e+05
2,2,6.028782e+05
3,3,2.278038e+05
4,4,3.713176e+05
